In [31]:
import math

# Fit alpha từ data thực đo
def fit_alpha(m1, m2, T1, T2):
    return math.log(T2 / T1) / math.log(m2 / m1)

alpha_avg = fit_alpha(1, 2, 80, 120)   # = 0.585
alpha_max = fit_alpha(1, 4, 80, 320)   # = 1.0  (T_max tăng tuyến tính)

def T_avg_est(m): return 80 * (m ** alpha_avg)
def T_max_est(m): return 80 * (m ** alpha_max)   # = 80 * m

print(T_avg_est(4))  # = 80 * 3^0.585 = 80 * 1.93 ≈ 154s
print(T_max_est(4))  # = 80 * 3       = 240s

180.0
320.0


In [3]:
import math
import numpy as np
from scipy.optimize import curve_fit

# ── Config ────────────────────────────────────────────────────────────────────
WARMUP_RUNS         = 0
M_MAX_EXTRAPOLATION = 2
M_MAX_FROM_VRAM     = 12
N_SIM               = 1000

# ── Data thực đo ──────────────────────────────────────────────────────────────
raw = {
    1:  [[107],[19],[24],[26],[22],[21],[12],[20],[14],[22]],
    2:  [[156,36],[24,37],[36,51],[13,23],[14,55]],
    6:  [[208,52,24,70,112,121],[109,112,110,113]],
    8:  [[217,53,25,71,113,71,131,131],[112,110]],
    10: [[202,71,24,72,36,79,83,84,91,96]],
}

class OCRBatchOptimizer:
    def __init__(self, raw, warmup=WARMUP_RUNS, m_max_extrap=M_MAX_EXTRAPOLATION,
                 m_vram=M_MAX_FROM_VRAM, n_sim=N_SIM):
        self.raw      = raw
        self.warmup   = warmup
        self.n_sim    = n_sim
        self.m_max_measured = max(raw.keys())
        self.m_hard_cap     = min(self.m_max_measured * m_max_extrap, m_vram)
        self.profile  = self._summarize()
        self.model_avg = self._fit("avg")
        self.model_max = self._fit("max")

    # ── 1. Benchmark ─────────────────────────────────────────────────────────
    def _summarize(self):
        profile = {}
        print("── 1. Benchmark Analysis ────────────────────────────────────────")
        for m, runs in self.raw.items():
            stable    = runs[self.warmup:]
            all_times = [t for run in stable for t in run]

            T_avg = np.mean(all_times)

            # std_item: std của từng item riêng lẻ — dùng cho pool simulation
            # Đây là variance thực của job times, phản ánh sự khác nhau giữa các tài liệu
            std_item = np.std(all_times, ddof=1) if len(all_times) > 1 else 0.0

            # std_run: std của T_max per run — dùng cho T_max_p95 (run-to-run stability)
            T_max_per_run = [max(run) for run in stable]
            T_max    = np.mean(T_max_per_run)
            std_run  = np.std(T_max_per_run, ddof=1) if len(stable) > 1 else 0.0

            cv = std_run / T_max if T_max > 0 else 0.0
            profile[m] = {
                "T_avg":    T_avg,
                "T_max":    T_max,
                "std_run":  std_run,   # run-to-run std của T_max  → dùng cho T_max_p95
                "std_item": std_item,  # std của individual items  → dùng cho pool sampling
                "n":        len(stable),
            }
            status = "✓" if cv < 0.05 else f"⚠ cv={cv*100:.1f}%"
            print(f"  m={m:2d}: T_avg={T_avg:6.1f} | T_max={T_max:6.1f} | "
                  f"std_run=±{std_run:5.1f} | std_item=±{std_item:5.1f} | {status}")
        return profile

    # ── 2. Power law fit ──────────────────────────────────────────────────────
    def _power_law(self, m, t0, alpha):
        return t0 * (m ** alpha)

    def _fit(self, kind):
        ms = np.array(list(self.profile.keys()), dtype=float)
        ts = np.array([v[f"T_{kind}"] for v in self.profile.values()])
        ws = np.array([math.sqrt(v["n"]) for v in self.profile.values()])
        popt, pcov = curve_fit(self._power_law, ms, ts,
                               p0=[100, 0.75], bounds=(0, [1e5, 2.0]),
                               sigma=1/ws, absolute_sigma=False)
        t0, alpha   = popt
        alpha_std   = math.sqrt(pcov[1,1]) if pcov is not None else 0.1
        pred = self._power_law(ms, *popt)
        r2   = 1 - np.sum((ts - pred)**2) / np.sum((ts - ts.mean())**2)
        model = {"t0": float(t0), "alpha": float(alpha),
                 "alpha_std": float(alpha_std), "r2": float(r2)}
        if model["r2"] < 0.92 or model["alpha"] > 1.3 or model["alpha"] < 0.4:
            print(f"⚠️  WARNING: Power-law fit kém cho {kind} "
                  f"(R²={model['r2']:.3f}, alpha={model['alpha']:.3f})")
        return model

    # ── Estimators ────────────────────────────────────────────────────────────
    def T_avg(self, m):
        if m in self.profile:
            return self.profile[m]["T_avg"]
        # Nếu T_avg fit kém (R² < 0.92), dùng T_max model làm upper bound
        # thay vì extrapolate từ fit không đáng tin.
        # T_avg luôn <= T_max nên estimate T_avg = T_max * (T_avg/T_max ratio trung bình)
        if self.model_avg["r2"] < 0.92 and self.model_max["r2"] >= 0.92:
            t_max_pred = self.model_max["t0"] * (m ** self.model_max["alpha"])
            # Tính ratio T_avg/T_max từ data đo thực (trung bình có trọng số theo n)
            ratios = [v["T_avg"] / v["T_max"] for v in self.profile.values() if v["T_max"] > 0]
            ratio  = float(np.mean(ratios))
            return t_max_pred * ratio
        return self.model_avg["t0"] * (m ** self.model_avg["alpha"])

    def T_max(self, m):
        return self.profile[m]["T_max"] if m in self.profile \
               else self.model_max["t0"] * (m ** self.model_max["alpha"])

    def T_max_p95(self, m):
        unc = (self.model_max["alpha_std"] / self.model_max["alpha"]
               if self.model_max["alpha"] > 0 else 0.2)
        if m in self.profile:
            std_run = self.profile[m]["std_run"]
            if std_run > 0:
                # Trường hợp bình thường: đủ runs để tính std thực
                return self.profile[m]["T_max"] + 1.645 * std_run
            else:
                # Chỉ 1 run → std_run=0, không có confidence interval thực.
                # Fallback: dùng model uncertainty (alpha_std) như với extrapolated m,
                # tránh việc m đo 1 lần trông "an toàn hơn" m nội suy từ nhiều điểm.
                return self.profile[m]["T_max"] * (1 + 1.645 * unc)
        # Extrapolated m: dùng model uncertainty
        t_pred = self.model_max["t0"] * (m ** self.model_max["alpha"])
        return t_pred * (1 + 1.645 * unc)

    # ── Penalty ───────────────────────────────────────────────────────────────
    def penalty(self, m):
        if m <= self.m_max_measured:
            return 0.0
        dist = math.log(m / self.m_max_measured) / math.log(2)
        unc  = self.model_max["alpha_std"] / self.model_max["alpha"] \
               if self.model_max["alpha"] > 0 else 0.2
        mem_ratio  = m / self.m_hard_cap
        saturation = 0.18 * (mem_ratio - 0.7)**2 if mem_ratio > 0.7 else 0.0
        return 0.025 + 0.035 * dist + 0.18 * unc + saturation

    # ── Batch model ───────────────────────────────────────────────────────────
    def calculate_metrics_batch(self, N, m, use_p95=False):
        k     = math.ceil(N / m)
        r     = N % m
        t_avg = self.T_avg(m)
        t_max = self.T_max_p95(m) if use_p95 else self.T_max(m)
        makespan = k * t_max
        tput     = N / makespan
        if r == 0:
            avg_lat = t_avg + t_max * (k - 1) / 2
        else:
            lat_full = t_avg + t_max * (k - 2) / 2 if k > 1 else t_avg
            sum_full = (k - 1) * m * lat_full
            lat_last = t_avg + t_max * (k - 1)
            sum_last = r * lat_last
            avg_lat  = (sum_full + sum_last) / N
        return avg_lat, makespan, tput

    # ── Worker pool model ─────────────────────────────────────────────────────
    def _sample_job_times(self, m, n_jobs):
        """
        Sample individual job times từ log-normal distribution.

        FIX: Dùng std_item (std của từng item riêng lẻ trong batch) thay vì
        std_run (std của T_max per run). std_item phản ánh đúng variance của
        thời gian xử lý từng tài liệu — đây là nguồn gốc variance trong pool.

        std_run chỉ đo run-to-run stability của batch worst-case (T_max),
        không đo variance giữa các documents trong cùng 1 batch.
        """
        mu  = self.T_avg(m)
        # ✅ Dùng std_item thay vì std_run (đây là bug cũ)
        std = self.profile[m]["std_item"] if m in self.profile \
              else mu * 0.15  # fallback: 15% CV khi phải extrapolate

        if std <= 0 or mu <= 0:
            return np.full(n_jobs, mu)

        # Log-normal: fit từ mean và std thực đo
        cv2    = (std / mu) ** 2
        sigma2 = math.log(1 + cv2)
        mu_ln  = math.log(mu) - sigma2 / 2
        return np.random.lognormal(mu_ln, math.sqrt(sigma2), n_jobs)

    def calculate_metrics_pool(self, N, m, use_p95=False, n_sim=None):
        """
        Worker pool: slot free ngay khi job xong → job tiếp theo vào luôn.
        Monte Carlo vì job times có variance (tài liệu khác độ dài).

        Edge case N <= m: tất cả jobs chạy song song ngay từ đầu, không có
        scheduling → kết quả giống hệt 1 batch đơn. Monte Carlo sẽ cho kết quả
        sai vì resample từ distribution thay vì dùng giá trị thực đo.
        """
        if n_sim is None:
            n_sim = self.n_sim

        # Khi N <= m: không cần Monte Carlo
        if N <= m:
            makespan = self.T_max_p95(m) if use_p95 else self.T_max(m)
            avg_lat  = self.T_avg(m)
            return avg_lat, makespan, N / makespan

        avg_lats  = []
        makespans = []

        for _ in range(n_sim):
            job_times    = self._sample_job_times(m, N)
            worker_free  = np.zeros(m)
            finish_times = []

            for t in job_times:
                w              = int(np.argmin(worker_free))
                start          = worker_free[w]
                end            = start + t
                worker_free[w] = end
                finish_times.append(end)

            finish_times = np.array(finish_times)
            avg_lats.append(float(np.mean(finish_times)))
            makespans.append(float(np.max(finish_times)))

        if use_p95:
            avg_lat  = float(np.percentile(avg_lats,  95))
            makespan = float(np.percentile(makespans, 95))
        else:
            avg_lat  = float(np.mean(avg_lats))
            makespan = float(np.mean(makespans))

        return avg_lat, makespan, N / makespan

    # ── Find optimal ─────────────────────────────────────────────────────────
    def find_optimal(self, N, use_p95=False, mode="pool"):
        mode_str = f"{mode}  {'p95' if use_p95 else 'avg'}"
        print(f"\n── N={N}  M_HARD_CAP={self.m_hard_cap}  mode={mode_str} {'─'*28}")
        print(f"{'m':>4} | {'T_avg':>7} | {'T_max':>7} | {'p95':>7} | "
              f"{'avg_lat':>9} | {'makespan':>9} | {'tput':>10} | {'penalty':>7} | {'src':>8}")
        print("-" * 95)

        results = []
        for m in range(1, min(N, self.m_hard_cap) + 1):
            if mode == "pool":
                lat, ms, tput = self.calculate_metrics_pool(N, m, use_p95)
            else:
                lat, ms, tput = self.calculate_metrics_batch(N, m, use_p95)

            pen     = self.penalty(m)
            eff_lat = lat * (1 + pen)
            src_str = "đo thực" if m in self.profile else "nội suy"
            pen_str = f"{pen*100:.1f}%" if pen > 0 else "   ---"
            p95_val = self.T_max_p95(m)
            p95_str = f"{p95_val:.1f}"

            print(f"{m:>4} | {self.T_avg(m):>5.1f}  | {self.T_max(m):>5.1f}  | "
                  f"{p95_str:>7} | {lat:>7.1f}  | {ms:>7.0f}  | "
                  f"{tput:>9.4f}/s | {pen_str:>7} | {src_str:>8}")

            results.append((m, eff_lat, lat, ms, tput))

        best = min(results, key=lambda x: x[1])
        print(f"\n→ OPTIMAL m* = {best[0]} | eff_lat={best[1]:.1f} | "
              f"avg_lat={best[2]:.1f} | makespan={best[3]:.0f} | tput={best[4]:.4f}/s")
        return best[0]

    def summary(self):
        print(f"\n── 2. Power Law Fit ─────────────────────────────────────────────")
        for kind, model in [("T_avg", self.model_avg), ("T_max", self.model_max)]:
            print(f"  {kind}(m) = {model['t0']:.2f} × m^{model['alpha']:.4f} "
                  f"(R²={model['r2']:.4f}, α_std=±{model['alpha_std']:.3f})")
        print(f"\n── 3. Giới hạn m ────────────────────────────────────────────────")
        print(f"  m_max đo thực = {self.m_max_measured}")
        print(f"  M_HARD_CAP    = {self.m_hard_cap}")

# ── Chạy ─────────────────────────────────────────────────────────────────────
opt = OCRBatchOptimizer(raw)
opt.summary()

for N in [8, 16, 32]:
    opt.find_optimal(N, use_p95=True, mode="batch")
    opt.find_optimal(N, use_p95=True, mode="pool")

── 1. Benchmark Analysis ────────────────────────────────────────
  m= 1: T_avg=  28.7 | T_max=  28.7 | std_run=± 27.8 | std_item=± 27.8 | ⚠ cv=97.0%
  m= 2: T_avg=  44.5 | T_max=  64.4 | std_run=± 52.7 | std_item=± 41.6 | ⚠ cv=81.9%
  m= 6: T_avg= 103.1 | T_max= 160.5 | std_run=± 67.2 | std_item=± 49.0 | ⚠ cv=41.9%
  m= 8: T_avg= 103.4 | T_max= 164.5 | std_run=± 74.2 | std_item=± 53.2 | ⚠ cv=45.1%
  m=10: T_avg=  83.8 | T_max= 202.0 | std_run=±  0.0 | std_item=± 47.6 | ✓
⚠️  WARNING: Power-law fit kém cho avg (R²=0.748, alpha=0.562)

── 2. Power Law Fit ─────────────────────────────────────────────
  T_avg(m) = 30.78 × m^0.5619 (R²=0.7481, α_std=±0.111)
  T_max(m) = 34.42 × m^0.7832 (R²=0.9702, α_std=±0.084)

── 3. Giới hạn m ────────────────────────────────────────────────
  m_max đo thực = 10
  M_HARD_CAP    = 12

── N=8  M_HARD_CAP=12  mode=batch  p95 ────────────────────────────
   m |   T_avg |   T_max |     p95 |   avg_lat |  makespan |       tput | penalty |      src
----------